### Update the `year`, the `timeframe` and the `username` below to reflect the relevant details. 

These will impact the Folder, Filenames, the Email Subject & Body, and the BigQuery table where the results are saved.</br></br>
Examples below:</br></br>
**Folder:** Shared Documents\Enrichment\Vendor\Vendor Scorecards\2025\Q1  
**File:** NESTLE_PCX_ENRICHMENT_SCORECARD_Q1_2025.xlsx  
**Email Subject:** PCX Supplier Enrichment Scorecard - Nestle Q1-2025  
**Email Body:**  We are pleased to send you your PCX Supplier Enrichment Scorecard for Q1 2025.  
**BQ Table:** `ld-pcx-bia.Merch_PIM.2025Q1_SUPPLIER_SCORECARD_ITEMS`

### Time period to be used in the filename e.g. 'Q1' for first quarter or '01' for January or 'WK1' for week 1

In [43]:
timeframe = 'Q2'
year = '2025'

### Username of the person running the query. E.g. the username used for your LCL login such as` jongrub` in C:\Users\\`jongrub`\OneDrive - George Weston Limited-6469347-MTCAD\Shared Documents

In [44]:
username = 'jongrub'

In [45]:
# Mark the time the script started running

import datetime

starttime = datetime.datetime.now()

print('This script starting running at ' + str(starttime))

This script starting running at 2025-07-02 14:18:03.976000


In [46]:
# to install any packages, remove comment from that line and run this cell

#pip install google-cloud-bigquery 
#pip install pyarrow 
#pip install pandas 
#pip install db-dtypes
#pip install XlWings 

#pip install Office365-REST-Python-Client 

In [47]:
# Set the directory to the Vendor Scorecards Folder

import os
from pathlib import Path


# Set the root folder
root_folder = f'C:\\Users\\{username}\\OneDrive - George Weston Limited-6469347-MTCAD\\Shared Documents\\Enrichment\\Vendor\\Vendor Scorecards'

# Refer to the latest template file
template_file = f'{root_folder}\\PCX_ENRICHMENT_SCORECARD - TEMPLATE.xlsx'

# Create new folders for this year and timeframe if they don't already exist
new_directory = f'{root_folder}\\{year}\\{timeframe}'
Path(new_directory).mkdir(parents=True, exist_ok=True)

# Set the current working directory to the folder for the current year/timeframe
os.chdir(new_directory)
cwd = os.getcwd()
print('The current working directory is ' + '"' + cwd + '"')

The current working directory is "C:\Users\jongrub\OneDrive - George Weston Limited-6469347-MTCAD\Shared Documents\Enrichment\Vendor\Vendor Scorecards\2025\Q2"


In [ ]:
# Connect to BigQuery 
from google.cloud import bigquery
import google.auth
project = 'ld-pcx-bia'
conn = bigquery.Client(project=project)

# Run query to create a table for all scorecard items for the current run
# Bring that created table into python

query = f'''
    CREATE OR REPLACE TABLE `ld-pcx-bia.Merch_PIM.{year}{timeframe}_SUPPLIER_SCORECARD_ITEMS`
    AS  SELECT * FROM `ld-pcx-bia.Merch_PIM.supplier_scorecard_v`; 
    
    SELECT * FROM `ld-pcx-bia.Merch_PIM.{year}{timeframe}_SUPPLIER_SCORECARD_ITEMS`
    '''

conn.query(
                    query,
                    bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
                ).total_bytes_billed
query_result = conn.query_and_wait(query)
query_result = conn.query(query)

# Create a dataframe with data for all suppliers
df = query_result.to_dataframe()

# Sort dataframe by the relevant columns
df = df.sort_values(['rolodex_name','sort_order'], ignore_index=True)

# print a preview of the data for all suppliers
df.head()

In [51]:
import xlwings as xw
import pandas as pd

# Create a list of all unique suppliers mentioned in the table
unique_vendors = df['rolodex_name'].unique()

# Create a connected instance to Excel
with xw.App(visible = True) as app:
    # By default will open with a new workbook called Book1 - ensure there is no Book1 already open
    book = app.books['Book1']
    # Loop through each unique vendor
    for vendor in unique_vendors:
        # Filter the dataframe for the current vendor, excluding the rolodex_name column
        vendor_df = df[df['rolodex_name'] == vendor].drop(columns=['rolodex_name','sort_order'])

        # Construct the xlsx filename based on the vendor name and timeframe
        vendor_filename = f"{cwd}\{vendor}_PCX_SUPPLIER_ENRICHMENT_SCORECARD_{timeframe}_{year}.xlsx"
        
        # Create a subtitle with the supplier name and timeframe
        file_subtitle = f"{vendor} {timeframe} {year}"

        # Open the template file 
        wb = xw.Book(template_file)

        # Select the Item Details worksheet
        ws = wb.sheets['ITEM DETAILS']

        # Paste the data from the vendor's dataframe into the selected tab of the template file starting at cell A4
        ws["A4"].options(pd.DataFrame, header=0, index=False, expand='vertical').value = vendor_df
        
        # Paste the subtitle into cell A1 on the Item Details tab
        ws["A1"].value = file_subtitle

        # Select the Item Details worksheet
        ws = wb.sheets['SUMMARY']
        
        # Change the active tab to the Summary tab before refreshing the file to ensure the charts update correctly
        wb.sheets['SUMMARY']
        wb.api.RefreshAll()
        
        # Paste the subtitle into cell L3 on the Summary tab
        ws["L3"].value = file_subtitle
        
        # Change the active sheet to what the file should open with
        wb.sheets['SCORING & INSTRUCTIONS']

        # Save the template file with a new name
        wb.save(vendor_filename)

        # Close the workbook
        wb.close()

In [50]:
# Mark the time the script stopped running

import datetime

endtime = datetime.datetime.now()

print('This script stopped running at ' + str(endtime))

This script stopped running at 2025-07-02 14:25:12.144221
